In [ ]:
pip install pandas tqdm

In [ ]:
import pandas as pd

# Load Excel file
df = pd.read_excel("DhoroniDataset.xlsx")

print("Total articles:", len(df))
print(df.head())


Total articles: 2300
                    dates                                              Title  \
0     18 April 2021 11:49                      ঢাকায় বায়ু দূষণ রোধে করণীয়   
1  26 November 2019 04:34  বায়ু দূষণ কমাতে ঢাকাসহ ৫ জেলায় ইটভাটা বন্ধের...   
2  20 February 2024 08:07  ঝুঁকিপূর্ণ বাতাস: ঢাকায় ঘরের বাইরে মাস্ক পরার...   
3      08 June 2022 18:22  পরিবেশের স্বাস্থ্যরক্ষায় সব শেষে ভারত! অনুমানে...   
4      06 June 2022 08:32     পরিবেশ রক্ষায় বহুমাত্রিক প্রচেষ্টা, দাবি মোদীর   

          File Political Influence Scientific Data Mentioned  \
0     ID-1.txt                  No                       Yes   
1    ID-10.txt                  No                        No   
2   ID-100.txt                  No                        No   
3  ID-1000.txt                  No                        No   
4  ID-1003.txt                 Yes                        No   

  Statistical Data Mentioned Source Mentioned Authenticity Stance Detection  \
0                        Yes      

In [ ]:
import pandas as pd
from google.colab import files

# Step 1 — Load cleaned dataset
df = pd.read_excel("DhoroniDataset_cleaned.xlsx")

print("Columns in dataset:", df.columns.tolist())

# Step 2 — Merge Title + File into a single 'text' field
# Note: 'File' contains the article content
df['text'] = "Title: " + df['Title'].astype(str) + "\nArticle: " + df['File'].astype(str)

# Step 3 — Keep only relevant columns
# We will keep 'id' if exists, else generate one
if 'id' not in df.columns:
    df['id'] = range(1, len(df)+1)  # simple numeric ID

df = df[['id', 'Focus', 'text']]  # labels can be added later if needed

print(df.head())

# Step 4 — Save merged dataset
merged_file = "DhoroniDataset_merged.xlsx"
df.to_excel(merged_file, index=False)
print(f"Merged dataset saved as {merged_file}")

# Step 5 — Download the merged dataset
files.download(merged_file)

Columns in dataset: ['dates', 'Title', 'File', 'Source Mentioned', 'Authenticity', 'Stance Detection', 'Focus']
   id                     Focus  \
0   1             Air Pollution   
1   2             Air Pollution   
2   3             Air Pollution   
3   4  Environmental Protection   
4   5  Environmental Protection   

                                                text  
0  Title: ঢাকায় বায়ু দূষণ রোধে করণীয়\nArticle:...  
1  Title: বায়ু দূষণ কমাতে ঢাকাসহ ৫ জেলায় ইটভাটা...  
2  Title: ঝুঁকিপূর্ণ বাতাস: ঢাকায় ঘরের বাইরে মাস...  
3  Title: পরিবেশের স্বাস্থ্যরক্ষায় সব শেষে ভারত! ...  
4  Title: পরিবেশ রক্ষায় বহুমাত্রিক প্রচেষ্টা, দাব...  
Merged dataset saved as DhoroniDataset_merged.xlsx


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>


**CONVERTING MERGE DATASET INTO "JSONL"**:

In [ ]:
import pandas as pd
import json
from tqdm import tqdm
from google.colab import files

# Step 1 — Load the merged dataset
df = pd.read_excel("DhoroniDataset_merged.xlsx")
print("Columns in dataset:", df.columns.tolist())

# Step 2 — Define output JSONL file
jsonl_file = "DhoroniDataset_for_LLM.jsonl"

# Step 3 — Loop through each row and write as JSON
# Each line in JSONL is a separate JSON object for one article
with open(jsonl_file, "w", encoding="utf-8") as f:
    for _, row in tqdm(df.iterrows(), total=len(df)):
        record = {
            "id": row["id"],
            "focus": row["Focus"],  # keep focus label
            "text": row["text"],
            "labels": []  # initially empty; will be filled after LLM cause extraction
        }
        f.write(json.dumps(record, ensure_ascii=False) + "\n")

print(f"JSONL file saved as {jsonl_file}")

# Step 4 — Download the JSONL file
files.download(jsonl_file)

Columns in dataset: ['id', 'Focus', 'text']


100%|██████████| 2300/2300 [00:00<00:00, 8803.22it/s]

JSONL file saved as DhoroniDataset_for_LLM.jsonl


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
import pandas as pd
import json

# Path to your uploaded JSONL file
input_jsonl = "DhoroniDataset_for_LLM.jsonl"

# Load JSONL into a list of dictionaries
articles = []
with open(input_jsonl, "r", encoding="utf-8") as f:
    for line in f:
        articles.append(json.loads(line))

# Check how many articles loaded
print(f"Total articles loaded: {len(articles)}")

# Inspect the first article
print(json.dumps(articles[0], ensure_ascii=False, indent=2))


Total articles loaded: 2300
{
  "id": 1,
  "focus": "Air Pollution",
  "text": "Title: ঢাকায় বায়ু দূষণ রোধে করণীয়\nArticle: ID-1.txt",
  "labels": []
}


Step 0: Install dependencies

In [ ]:
!pip install transformers datasets torch sentencepiece --quiet


Step 1: Load your uploaded JSONL

In [ ]:
import pandas as pd

# If you uploaded from PC, it should be in /content
input_file = "DhoroniDataset_for_LLM.jsonl"

df = pd.read_json(input_file, lines=True)
print("Loaded rows:", len(df))
df.head()


Loaded rows: 2300


,id,focus,text,labels
0,1,Air Pollution,Title: ঢাকায় বায়ু দূষণ রোধে করণীয়\nArticle:...,[]
1,2,Air Pollution,Title: বায়ু দূষণ কমাতে ঢাকাসহ ৫ জেলায় ইটভাটা...,[]
2,3,Air Pollution,Title: ঝুঁকিপূর্ণ বাতাস: ঢাকায় ঘরের বাইরে মাস...,[]
3,4,Environmental Protection,Title: পরিবেশের স্বাস্থ্যরক্ষায় সব শেষে ভারত! ...,[]
4,5,Environmental Protection,"Title: পরিবেশ রক্ষায় বহুমাত্রিক প্রচেষ্টা, দাব...",[]


Step 2: Define Manual Cause List and Multi-hot Mapping

In [ ]:
# Flatten all causes from all focuses into a single total list
all_causes = sorted(list({cause for causes in focus2causes.values() for cause in causes}))
print("Total unique causes:", len(all_causes))
print(all_causes)

# Create mappings for multi-hot vectors
cause2idx = {cause: i for i, cause in enumerate(all_causes)}
idx2cause = {i: cause for cause, i in cause2idx.items()}


Total unique causes: 33
['advertisement_billboards', 'agricultural_burning', 'agrochemical_overuse', 'arsenic_contamination', 'brick_kilns', 'climate_change', 'coal_power_plants', 'construction_and_road_dust', 'construction_noise', 'deforestation_land_use_change', 'fossil_fuel_burning', 'glacier_melting', 'global_warming', 'habitat_loss', 'heavy_metals_contamination', 'illegal_logging', 'illegal_poaching', 'industrial_emissions', 'industrial_lighting', 'industrial_noise', 'industrial_waste_dumping', 'industrial_wastewater_discharge', 'land_use_change', 'landfill_leachate', 'loudspeakers_and_social_noise', 'open_waste_burning', 'plastic_waste_in_water', 'sea_level_rising', 'sewage_and_drainage', 'solid_waste_dumping', 'traffic_noise', 'urban_street_lighting', 'vehicle_emissions']


Step 2b: Convert manual causes per article to multi-hot vectors

In [ ]:
# Step 2: Prepare BanglaBERT-ready dataset with manual causes
import pandas as pd
import numpy as np
import json

# ---------- 1. Load JSONL ----------
input_file = "/content/DhoroniDataset_for_LLM .jsonl"  # Adjust path if needed
df = pd.read_json(input_file, lines=True)

# ---------- 2. Clean column names ----------
df.columns = df.columns.str.strip()  # remove leading/trailing spaces
print("Columns in dataset:", df.columns.tolist())

# If the focus column is not exactly 'Focus', rename it
# Replace 'focus_column_name' with actual column name if different
if 'Focus' not in df.columns:
    possible_focus_cols = [c for c in df.columns if 'focus' in c.lower()]
    if possible_focus_cols:
        df.rename(columns={possible_focus_cols[0]: 'Focus'}, inplace=True)
    else:
        raise ValueError("No focus column found! Please check your dataset.")

# Check Focus values
print("Unique Focus values:", df["Focus"].unique())

# ---------- 3. Define manual cause list ----------
focus2causes = {
    "Air Pollution": [
        "vehicle_emissions",
        "industrial_emissions",
        "brick_kilns",
        "open_waste_burning",
        "construction_and_road_dust"
    ],
    "Plastic Pollution": [
        "plastic_waste_in_water",
        "open_waste_burning",
        "solid_waste_dumping"
    ],
    "Water Pollution": [
        "industrial_wastewater_discharge",
        "sewage_and_drainage",
        "arsenic_contamination",
        "plastic_waste_in_water"
    ],
    "Global Warming": [
        "fossil_fuel_burning",
        "deforestation_land_use_change",
        "industrial_emissions",
        "coal_power_plants"
    ],
    "Sound Pollution": [
        "traffic_noise",
        "industrial_noise",
        "construction_noise",
        "loudspeakers_and_social_noise"
    ],
    "Heavy Metal": [
        "industrial_emissions",
        "industrial_wastewater_discharge",
        "heavy_metals_contamination"
    ],
    "Climate Change": [
        "fossil_fuel_burning",
        "deforestation_land_use_change",
        "industrial_emissions",
        "agricultural_burning"
    ],
    "Light Pollution": [
        "urban_street_lighting",
        "industrial_lighting",
        "advertisement_billboards"
    ],
    "Water Resource Pollution": [
        "industrial_wastewater_discharge",
        "agrochemical_overuse",
        "sewage_and_drainage"
    ],
    "Sea Level Rising": [
        "global_warming",
        "glacier_melting",
        "deforestation_land_use_change"
    ],
    "Forestry and Tree Plantation": [
        "deforestation_land_use_change",
        "illegal_logging",
        "land_use_change"
    ],
    "Soil Pollution": [
        "agrochemical_overuse",
        "industrial_waste_dumping",
        "landfill_leachate"
    ],
    "Wildlife": [
        "deforestation_land_use_change",
        "habitat_loss",
        "illegal_poaching"
    ],
    "Rain": [
        "climate_change",
        "global_warming",
        "deforestation_land_use_change"
    ],
    "Natural Disaster": [
        "climate_change",
        "sea_level_rising",
        "deforestation_land_use_change"
    ]
}

# ---------- 4. Map manual causes per article ----------
df["manual_causes"] = df["Focus"].apply(lambda x: focus2causes.get(x, []))

# ---------- 5. Flatten all causes for multi-hot labels ----------
all_causes = sorted(list({c for causes in focus2causes.values() for c in causes}))
cause2idx = {cause: i for i, cause in enumerate(all_causes)}
idx2cause = {i: cause for cause, i in cause2idx.items()}

# Function to convert cause list to multi-hot vector
def causes_to_multihot(causes, mapping):
    vec = np.zeros(len(mapping), dtype=int)
    for c in causes:
        if c in mapping:
            vec[mapping[c]] = 1
    return vec

df["labels"] = df["manual_causes"].apply(lambda x: causes_to_multihot(x, cause2idx))

# ---------- 6. Optional: Add article text column for training ----------
if "text" in df.columns:
    df["article_text"] = df["text"]
else:
    raise ValueError("No column found for article text. Please check your dataset.")

# ---------- 7. Save BanglaBERT-ready JSONL ----------
output_file = "/content/Dhoroni_BanglaBERT_ready.jsonl"
df.to_json(output_file, orient="records", lines=True, force_ascii=False)
print("✅ Dataset saved for BanglaBERT training:", output_file)

Columns in dataset: ['id', 'focus', 'text', 'labels']
Unique Focus values: ['Air Pollution' 'Environmental Protection' 'Plastic Pollution'
 'Environmental Pollution' 'Water Pollution' 'Global Warming'
 'Sound Pollution' 'Heavy Metal' 'Climate Change' 'Light Pollution'
 'Forestry and Tree Plantation' 'Soil Pollution'
 'Water Resource Pollution' 'Sea Level Rising' 'Water Pollution '
 'Wildlife' 'E-Waste' 'Foresty and Tree Plantation' 'Rain'
 'Natural Disaster' 'Environmental  Protection' 'Ozone Layer']
✅ Dataset saved for BanglaBERT training: /content/Dhoroni_BanglaBERT_ready.jsonl


In [ ]:
import json

# Your manual focus2causes dictionary
focus2causes = {
    "Air Pollution": [
        "vehicle_emissions",
        "industrial_emissions",
        "brick_kilns",
        "open_waste_burning",
        "construction_and_road_dust"
    ],
    "Plastic Pollution": [
        "plastic_waste_in_water",
        "open_waste_burning",
        "solid_waste_dumping"
    ],
    "Water Pollution": [
        "industrial_wastewater_discharge",
        "sewage_and_drainage",
        "arsenic_contamination",
        "plastic_waste_in_water"
    ],
    "Global Warming": [
        "fossil_fuel_burning",
        "deforestation_land_use_change",
        "industrial_emissions",
        "coal_power_plants"
    ],
    "Sound Pollution": [
        "traffic_noise",
        "industrial_noise",
        "construction_noise",
        "loudspeakers_and_social_noise"
    ],
    "Heavy Metal": [
        "industrial_emissions",
        "industrial_wastewater_discharge",
        "heavy_metals_contamination"
    ],
    "Climate Change": [
        "fossil_fuel_burning",
        "deforestation_land_use_change",
        "industrial_emissions",
        "agricultural_burning"
    ],
    "Light Pollution": [
        "urban_street_lighting",
        "industrial_lighting",
        "advertisement_billboards"
    ],
    "Water Resource Pollution": [
        "industrial_wastewater_discharge",
        "agrochemical_overuse",
        "sewage_and_drainage"
    ],
    "Sea Level Rising": [
        "global_warming",
        "glacier_melting",
        "deforestation_land_use_change"
    ],
    "Forestry and Tree Plantation": [
        "deforestation_land_use_change",
        "illegal_logging",
        "land_use_change"
    ],
    "Soil Pollution": [
        "agrochemical_overuse",
        "industrial_waste_dumping",
        "landfill_leachate"
    ],
    "Wildlife": [
        "deforestation_land_use_change",
        "habitat_loss",
        "illegal_poaching"
    ],
    "Rain": [
        "climate_change",
        "global_warming",
        "deforestation_land_use_change"
    ],
    "Natural Disaster": [
        "climate_change",
        "sea_level_rising",
        "deforestation_land_use_change"
    ]
}

# Bangla solutions mapping (example)
cause2solution_bangla = {
    "vehicle_emissions": "সার্বজনীন পরিবহন বৃদ্ধি করুন এবং ব্যক্তিগত গাড়ির ব্যবহার সীমিত করুন।",
    "industrial_emissions": "কারখানায় বাতাসের ফিল্টার ব্যবহার এবং নির্গমন নিয়ন্ত্রণ নীতি প্রয়োগ করুন।",
    "brick_kilns": "পরিবেশ বান্ধব ইটভাটা প্রযুক্তি ব্যবহার এবং বসতবাড়ি থেকে দূরে স্থানান্তর করুন।",
    "open_waste_burning": "খোলা বর্জ্য জ্বালানো বন্ধ করুন এবং বর্জ্য সংগ্রহের ব্যবস্থা করুন।",
    "construction_and_road_dust": "নির্মাণ স্থলে পানি ছিটিয়ে ধুলো কমান এবং রাস্তা ঢেকে রাখুন।",
    "plastic_waste_in_water": "প্লাস্টিক ব্যবস্থাপনা উন্নত করুন এবং পুনঃব্যবহারযোগ্য পণ্য ব্যবহার করুন।",
    "solid_waste_dumping": "সংগঠিত বর্জ্য নিষ্পত্তি ও পুনঃপ্রক্রিয়াকরণ কেন্দ্র তৈরি করুন।",
    "industrial_wastewater_discharge": "শিল্প কারখানার বর্জ্য জলে ফেলা নিয়ন্ত্রণ করুন।",
    "sewage_and_drainage": "পানি নিষ্কাশন ও নিকাশী ব্যবস্থার উন্নতি করুন।",
    "arsenic_contamination": "পানি পরীক্ষার ব্যবস্থা বৃদ্ধি করুন এবং আর্সেনিক দূষণ রোধ করুন।",
    "fossil_fuel_burning": "জ্বালানি দাহ কমান এবং নবায়নযোগ্য শক্তি ব্যবহার করুন।",
    "deforestation_land_use_change": "বন সংরক্ষণ করুন এবং অবৈধ চাষ ও কেটে ফেলা রোধ করুন।",
    "coal_power_plants": "কয়লা ব্যবহার কমান এবং পরিচ্ছন্ন শক্তি উৎস ব্যবহার করুন।",
    "traffic_noise": "যানবাহনের গতি নিয়ন্ত্রণ করুন এবং শহরে যানবাহন সংখ্যা কমান।",
    "industrial_noise": "শিল্প প্রতিষ্ঠানগুলিতে শব্দ নিয়ন্ত্রণ ব্যবস্থা ব্যবহার করুন।",
    "construction_noise": "নির্মাণ সময় নির্দিষ্ট ঘণ্টায় কাজ করুন এবং শব্দ কমানোর যন্ত্র ব্যবহার করুন।",
    "loudspeakers_and_social_noise": "সামাজিক ও জনসাধারণের শব্দ নিয়ন্ত্রণ করুন।",
    "heavy_metals_contamination": "শিল্প বর্জ্য নিয়ন্ত্রণ এবং ধাতু দূষণ রোধ করুন।",
    "agricultural_burning": "কৃষি জ্বালানি কমান এবং পরিবেশবান্ধব কৃষি পদ্ধতি অনুসরণ করুন।",
    "urban_street_lighting": "শহরের আলো নিয়ন্ত্রণ এবং অনাবশ্যক লাইট বন্ধ করুন।",
    "industrial_lighting": "শিল্প প্রতিষ্ঠানে আলো কমানো এবং দক্ষ আলো ব্যবস্থাপনা করুন।",
    "advertisement_billboards": "বিজ্ঞাপন বোর্ড কমান এবং পরিবেশ বান্ধব উপায় ব্যবহার করুন।",
    "glacier_melting": "গ্লেসিয়ারের সংরক্ষণ এবং জলবায়ু পরিবর্তন নিয়ন্ত্রণ।",
    "illegal_logging": "অবৈধ চুরি বন্ধ করুন এবং বন সংরক্ষণ বৃদ্ধি করুন।",
    "land_use_change": "ভূমি ব্যবহারের পরিকল্পনা করুন এবং বনাঞ্চল সংরক্ষণ করুন।",
    "agrochemical_overuse": "কৃষিতে রাসায়নিক ব্যবহার সীমিত করুন এবং জৈব চাষ প্রচার করুন।",
    "industrial_waste_dumping": "শিল্প বর্জ্য যথাযথভাবে নিষ্পত্তি করুন।",
    "landfill_leachate": "ল্যান্ডফিল থেকে দূষণ নিয়ন্ত্রণ করুন।",
    "habitat_loss": "প্রাণীর আবাসস্থল সংরক্ষণ করুন।",
    "illegal_poaching": "অবৈধ শিকার রোধ করুন।",
    "climate_change": "জলবায়ু পরিবর্তনের প্রভাব কমাতে ব্যবস্থা নিন।",
    "global_warming": "গ্লোবাল ওয়ার্মিং হ্রাসে উদ্যোগ নিন।",
    "sea_level_rising": "সমুদ্রপৃষ্ঠের উচ্চতা বৃদ্ধি রোধে ব্যবস্থা নিন।"
}

# ---------- Generate JSONL ----------
output_file = "/content/solution_framework.jsonl"

with open(output_file, "w", encoding="utf-8") as f:
    for focus, causes in focus2causes.items():
        for cause in causes:
            solution = cause2solution_bangla.get(cause, "উপযুক্ত সমাধান প্রদান করা হয়নি।")
            record = {
                "focus": focus,
                "cause": cause,
                "solution": solution
            }
            f.write(json.dumps(record, ensure_ascii=False) + "\n")

print("✅ Bangla solution framework JSONL saved:", output_file)


✅ Bangla solution framework JSONL saved: /content/solution_framework.jsonl


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

!cp /content/DhoroniDataset_for_LLM.jsonl /content/drive/MyDrive/softcomproject/
!cp /content/solution_framework.jsonl /content/drive/MyDrive/softcomproject/


Mounted at /content/drive
cp: cannot stat '/content/DhoroniDataset_for_LLM.jsonl': No such file or directory
cp: cannot create regular file '/content/drive/MyDrive/softcomproject/': Not a directory
